In [ ]:
# Connection to google drive
from google.colab import drive
drive.mount('/content/drive')

import json

Mounted at /content/drive


Create the first version of the spices map file that we will use to filter the spices from the recipes.

1.   First, load the spices JSON file we took from the SpiceNice site.
2.   Then, separate the spices into powder, dried, seeds, and add their possible alternative names.
3.   Finally, save the result as 'spices_map_temp.json'.

In [ ]:
# Read the spices JSON file (from SpiceNice site)
with open("/content/drive/MyDrive/Data Mining - Final Project/spices json/spices.json", encoding="utf-8") as f:
    spices_data = json.load(f)

spice_map = {}

# Forms we want to split into separate sub-spices
sub_spice_keywords = {
    "powder": ["powder", "ground"],
    "dried": ["dried"],
    "seeds": ["seed", "seeds"]
}

for spice in spices_data:
    base_name = spice["name"].strip().lower()
    all_names = [base_name]

    # Add alternative names
    if spice.get("alternativeNames"):
        for alt in spice["alternativeNames"].split(","):
            alt = alt.strip().lower()
            if alt:
                all_names.append(alt)

    # Track culinary forms
    forms = []
    if spice.get("culinaryForm"):
        forms = [form.strip().lower() for form in spice["culinaryForm"].split(",")]

    # Initialize sets for sub-spices
    plain_syns = set()
    powder_syns = set()
    dried_syns = set()
    seeds_syns = set()

    for name in all_names:
        plain_syns.add(name)

        for form in forms:
            # If the form contains a sub-spice keyword, we will handle separately
            if any(k in form for k in sub_spice_keywords["powder"]):
                if "powder" not in name and "ground" not in name:
                    powder_syns.add(f"{name} powder")
                    powder_syns.add(f"ground {name}")
            elif any(k in form for k in sub_spice_keywords["dried"]):
                if "dried" not in name:
                    dried_syns.add(f"dried {name}")
            elif any(k in form for k in sub_spice_keywords["seeds"]):
                if "seed" not in name and "seeds" not in name:
                    seeds_syns.add(f"{name} seeds")
                    seeds_syns.add(f"{name} seed")
            else:
                # If other culinary forms -> add to plain spice synonyms
                plain_syns.add(f"{name} {form}")

    # Add to spice_map (and sorting for convenience)
    spice_map[base_name] = sorted(plain_syns)
    if powder_syns:
        spice_map[f"{base_name} powder"] = sorted(powder_syns)
    if dried_syns:
        spice_map[f"dried {base_name}"] = sorted(dried_syns)
    if seeds_syns:
        spice_map[f"{base_name} seeds"] = sorted(seeds_syns)

# Save spice_map to json file
with open("/content/drive/MyDrive/Data Mining - Final Project/spices json/spices_map_temp.json", "w", encoding="utf-8") as f:
    json.dump(spice_map, f, ensure_ascii=False, indent=2)

print(f"spices_map_temp.json created with {len(spice_map)} spices")

spices_map_temp.json created with 160 spices


We got 160 spices.

After generating the initial file, we manually reviewed it to ensure that everything was created correctly and logically.
We then little organized and cleaned the data, and finally saved the resulting spices mapping dictionary as **spices_map.json**.

In [ ]:
# Load the final spices mapping after cleaning
with open('/content/drive/MyDrive/Data Mining - Final Project/spices json/spices_map.json') as f:
    final_spice_map = json.load(f)

print(f"spices_map.json with {len(final_spice_map)} spices")

spices_map.json with 106 spices


In total, we received **106 spices** for spices mapping, which we will use to analyze the spices in recipes.